# Medical Insurance Cost Prediction
**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn, Scikit-learn

This notebook demonstrates a regression workflow for estimating insurance charges.

**Dataset notice:** The included `insurance.csv` is a synthetic practice dataset created for this recreated portfolio project. It contains 1,338 rows and 7 columns; it is not the original dataset from the earlier college assignment and contains no real personal or medical records.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

## 1. Load and inspect data

In [ ]:
df = pd.read_csv("insurance.csv")
print("Shape:", df.shape)
display(df.head())
display(df.isnull().sum())
print("Duplicate rows:", df.duplicated().sum())
display(df.describe(include="all"))

## 2. Exploratory data analysis

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df["charges"], kde=True)
plt.title("Distribution of Insurance Charges")
plt.show()

plt.figure(figsize=(8,5))
sns.scatterplot(data=df, x="age", y="charges", hue="smoker", alpha=.7)
plt.title("Charges by Age and Smoking Status")
plt.show()

plt.figure(figsize=(7,5))
sns.boxplot(data=df, x="smoker", y="charges")
plt.title("Charges by Smoking Status")
plt.show()

## 3. Prepare training and testing data

In [ ]:
X = df.drop(columns=["charges"])
y = df["charges"]
numeric = ["age", "bmi", "children"]
categorical = ["sex", "smoker", "region"]
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical)
])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

## 4. Train and compare regression models

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Lasso Regression": Lasso(alpha=100.0, max_iter=10000)
}
results = []
fitted = {}
for name, model in models.items():
    pipe = Pipeline([("preprocessor", preprocessor), ("model", model)])
    cv = cross_val_score(pipe, X_train, y_train, cv=5, scoring="r2")
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    results.append({
        "Model": name,
        "Mean 5-Fold CV R2": cv.mean(),
        "Test R2": r2_score(y_test, pred),
        "Test MAE": mean_absolute_error(y_test, pred),
        "Test RMSE": np.sqrt(mean_squared_error(y_test, pred))
    })
    fitted[name] = pipe
results_df = pd.DataFrame(results).sort_values("Mean 5-Fold CV R2", ascending=False)
display(results_df)

## 5. Select a model and inspect predictions

In [ ]:
best_name = results_df.iloc[0]["Model"]
best_model = fitted[best_name]
predictions = best_model.predict(X_test)
print("Selected model:", best_name)

plt.figure(figsize=(7,5))
plt.scatter(y_test, predictions, alpha=.65)
low = min(y_test.min(), predictions.min())
high = max(y_test.max(), predictions.max())
plt.plot([low, high], [low, high], "--")
plt.xlabel("Actual charges")
plt.ylabel("Predicted charges")
plt.title("Actual vs. Predicted Charges")
plt.show()

## 6. Make a sample prediction

In [ ]:
sample = pd.DataFrame([{
    "age": 35, "sex": "female", "bmi": 28.5,
    "children": 1, "smoker": "no", "region": "northwest"
}])
estimate = best_model.predict(sample)[0]
print(f"Estimated insurance charge: ${estimate:,.2f}")

## Conclusion
This project demonstrates data inspection, visualization, preprocessing, regression modelling, cross-validation, and evaluation. All model metrics are calculated from the included synthetic practice dataset when the notebook runs; they may differ from the earlier college assignment results.